# Báo cáo Kỹ thuật: Phân tích và Kiểm thử Ma trận (Part 1)

Notebook này trình diễn các thuật toán lõi của Part 1 bao gồm Khử Gauss, tính Định thức, Nghịch đảo và Hạng ma trận. 
Mục tiêu cốt lõi:
1. Xác minh tính đúng đắn bằng phương pháp **Assertion-based Testing**.
2. Phân tích tính ổn định số học qua **Ill-conditioned Matrices**.

In [7]:
from pathlib import Path
import numpy as np
import math

# Tự động cập nhật thay đổi từ các file .py
%load_ext autoreload
%autoreload 2

# Import các module tự viết
from gaussian import gaussian_eliminate, back_substitution
from determinant import determinant
from inverse import inverse
from rank_basis import rank_and_basis

print("✅ Môi trường đã sẵn sàng.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Môi trường đã sẵn sàng.


## 1. Cơ chế Kiểm chứng (Verification Logic)

Chúng ta sử dụng hàm `verify_system` để kiểm tra nghiệm $x$ của hệ $Ax = b$. 
Sai số được tính bằng **Relative Residual**:
$$\text{Error} = \frac{\|Ax - b\|}{\|b\|}$$
Sử dụng câu lệnh `assert` để đảm bảo sai số không vượt quá $10^{-5}$. Nếu vượt quá, hệ thống sẽ dừng thực thi.

In [8]:
def verify_system(A, x, b, label="Case"):
    A_np = np.array(A, dtype=float)
    x_np = np.array(x, dtype=float)
    b_np = np.array(b, dtype=float)
    
    # Tính sai số Ax - b
    residual = np.dot(A_np, x_np) - b_np
    error = np.linalg.norm(residual)
    
    try:
        x_ref = np.linalg.solve(A_np, b_np)
        assert np.allclose(x_np, x_ref, atol=1e-5), f"❌ {label}: Nghiệm không khớp với Numpy!"
    except np.linalg.LinAlgError:
        # Nếu hệ vô định/vô nghiệm, kiểm tra xem x có thỏa mãn tối ưu không
        assert error < 1e-5 or len(x) != len(A), f"❌ {label}: Sai số quá lớn!"
    
    print(f"   [OK] {label}: Verified (Error: {error:.2e})")

## 2. Giải hệ phương trình (Gaussian Elimination)

Thuật toán sử dụng phương pháp **Partial Pivoting** để tìm phần tử trụ lớn nhất, giúp giảm thiểu sai số làm tròn. Chúng ta sẽ thử nghiệm trên các cấu hình ma trận khác nhau (Vuông, Chữ nhật, Suy biến).

In [9]:
print("--- Testing Gaussian Suite  ---")
gauss_test_cases = [
    {"name": "Regular 3x3", "A": [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], "b": [8, -11, -3]},
    {"name": "Identity 4x4 (Large)", "A": np.eye(4).tolist(), "b": [1, 2, 3, 4]},
    {"name": "Float Entries", "A": [[0.5, 0.2], [0.1, 0.4]], "b": [0.7, 0.5]},
    {"name": "Rectangular Wide", "A": [[1, 2, 3], [4, 5, 6]], "b": [6, 15]},
    {"name": "Singular/Zero Row", "A": [[1, 2, 3], [0, 0, 0], [4, 5, 6]], "b": [6, 0, 15]},
    {"name": "Duplicate Rows", "A": [[1, 1], [1, 1]], "b": [2, 2]}
]

for case in gauss_test_cases:
    M, x, swaps = gaussian_eliminate(case["A"], case["b"])
    verify_system(case["A"], x, case["b"], label=case["name"])

--- Testing Gaussian Suite  ---
   [OK] Regular 3x3: Verified (Error: 8.88e-16)
   [OK] Identity 4x4 (Large): Verified (Error: 0.00e+00)
   [OK] Float Entries: Verified (Error: 0.00e+00)
   [OK] Rectangular Wide: Verified (Error: 0.00e+00)
   [OK] Singular/Zero Row: Verified (Error: 0.00e+00)
   [OK] Duplicate Rows: Verified (Error: 0.00e+00)


## 2. Kiểm thử tính toán Định thức (Determinant)

Định thức ($\det(A)$) là một giá trị quan trọng để xác định một ma trận có khả nghịch hay không. 
Chúng ta sử dụng `math.isclose` để so sánh kết quả từ hàm `determinant` với `numpy.linalg.det` với sai số cho phép ($10^{-7}$).

In [ ]:
import math
import numpy as np

print("--- Testing Determinant Suite ---")

det_test_cases = [
    {"name": "Identity 3x3", "A": np.eye(3).tolist()},
    {"name": "Upper Triangular", "A": [[2, 3, 1], [0, 5, 2], [0, 0, 3]]},
    {"name": "Singular (Det=0)", "A": [[1, 2, 3], [4, 5, 6], [7, 8, 9]]},
    {"name": "Small Ill-conditioned", "A": [[1, 1], [1, 1.0001]]},
    {"name": "Random 3x3", "A": [[6, 1, 1], [4, -2, 5], [2, 8, 7]]},
    {"name": "Negative Entries", "A": [[-1, 2], [3, -4]]}
]

for case in det_test_cases:
    A = case["A"]
    # Gọi hàm từ project
    actual = determinant(A)
    # Lấy giá trị chuẩn từ Numpy
    expected_np = np.linalg.det(np.array(A))
    
    # Assert để dừng notebook ngay nếu kết quả sai
    assert math.isclose(actual, expected_np, abs_tol=1e-7), \
        f"❌ {case['name']} FAILED: Got {actual}, Expected {expected_np}"
    
    print(f"   [OK] {case['name']}: Det = {actual:.4f} (Verified)")

print("\n🚀 TOÀN BỘ TEST DETERMINANT ĐÃ PASS!")

## 3. Ma trận Nghịch đảo (Matrix Inverse)

Một ma trận vuông $A$ có nghịch đảo $A^{-1}$ khi và chỉ khi $\det(A) \neq 0$. 
Trong bài test này, chúng ta đặc biệt kiểm tra **đường lỗi (Error Path)**: Nếu truyền vào ma trận suy biến, hàm phải ném ra lỗi `ValueError`.

In [ ]:
print("--- Testing Matrix Inverse  ---")
inv_test_suite = [
    {"name": "Regular 2x2", "A": [[4, 7], [2, 6]], "singular": False},
    {"name": "Identity 3x3", "A": np.eye(3).tolist(), "singular": False},
    {"name": "Negative Entries 2x2", "A": [[-1, -2], [3, 4]], "singular": False},
    {"name": "Larger 4x4", "A": (np.eye(4)*2).tolist(), "singular": False},
    {"name": "Singular (ValueError)", "A": [[1, 2], [2, 4]], "singular": True},
    {"name": "Zero Matrix (ValueError)", "A": [[0, 0], [0, 0]], "singular": True}
]

for case in inv_test_suite:
    A = case["A"]
    if case["singular"]:
        try:
            inverse(A)
            assert False, f"❌ {case['name']}: Lẽ ra phải báo ValueError!"
        except ValueError:
            print(f"   [OK] {case['name']}: Bắt lỗi ma trận suy biến thành công.")
    else:
        A_inv = inverse(A)
        res = np.dot(np.array(A), np.array(A_inv))
        assert np.allclose(res, np.eye(len(A)), atol=1e-7), f"❌ {case['name']}: Nghịch đảo sai!"
        print(f"   [OK] {case['name']}: Nghịch đảo chính xác.")

--- Testing Matrix Inverse  ---
   [OK] Regular 2x2: Nghịch đảo chính xác.
   [OK] Identity 3x3: Nghịch đảo chính xác.
   [OK] Negative Entries 2x2: Nghịch đảo chính xác.
   [OK] Larger 4x4: Nghịch đảo chính xác.
   [OK] Singular (ValueError): Bắt lỗi ma trận suy biến thành công.
   [OK] Zero Matrix (ValueError): Bắt lỗi ma trận suy biến thành công.


## 4. Phân tích Hệ số điều kiện (Ill-conditioned Analysis)

Ma trận điều kiện xấu là ma trận cực kỳ nhạy cảm với các thay đổi nhỏ ở đầu vào. 
Ví dụ: Một thay đổi $0.0001$ ở vector $b$ dẫn đến sai lệch khổng lồ ở vector $x$. 
Đây là bài học quan trọng về **Độ ổn định số học**.

In [ ]:
print("--- Phân tích Hệ số điều kiện (Ill-conditioned Matrix) ---")

# Thiết lập ma trận A nhạy cảm
A_ill = [[1, 1], [1, 1.0001]]
b1 = [2, 2.0001]
b2 = [2, 2.0002] # Chỉ thay đổi 0.0001 so với b1

# Giải hai hệ phương trình
_, x1, _ = gaussian_eliminate(A_ill, b1)
_, x2, _ = gaussian_eliminate(A_ill, b2)

print(f"   [b1] -> Nghiệm x1: {x1}")
print(f"   [b2] -> Nghiệm x2: {x2}")

# Assertion: Chứng minh rằng nghiệm thay đổi rất lớn (sai lệch > 0.5) 
# dù đầu vào chỉ thay đổi cực nhỏ (0.0001)
diff = np.linalg.norm(np.array(x1) - np.array(x2))
assert diff > 0.5, f"❌ Lỗi: Hệ thống không thể hiện tính điều kiện xấu (Diff: {diff:.2f})"

print(f"✅ Minh chứng thành công: Độ lệch nghiệm là {diff:.2f} (Rất lớn so với độ lệch đầu vào 0.0001)")

--- Phân tích Hệ số điều kiện (Ill-conditioned Matrix) ---
   [b1] -> Nghiệm x1: [0.9999999999977796, 1.0000000000022204]
   [b2] -> Nghiệm x2: [0.0, 2.0]
✅ Minh chứng thành công: Độ lệch nghiệm là 1.41 (Rất lớn so với độ lệch đầu vào 0.0001)


## 5. Hạng và Không gian cơ sở (Rank & Basis)

Hàm `rank_and_basis` thực hiện biến đổi ma trận về dạng **RREF (Reduced Row Echelon Form)** để xác định:
1. **Rank (Hạng):** Số lượng các cột chứa phần tử trụ (pivot).
2. **Column Space:** Cơ sở được trích xuất từ các cột tương ứng trên ma trận gốc $A$.
3. **Null Space:** Tìm các vector nghiệm của hệ phương trình thuần nhất $Ax = 0$.

In [ ]:
print("--- Testing Rank & Basis Suite (5+ Cases) ---")
rb_test_suite = [
    {"name": "Full Rank 3x3", "A": [[1, 0, 0], [0, 1, 0], [0, 0, 1]], "exp_rank": 3},
    {"name": "Rank 1 (All dependent)", "A": [[1, 2], [2, 4]], "exp_rank": 1},
    {"name": "Zero Matrix 2x2", "A": [[0, 0], [0, 0]], "exp_rank": 0},
    {"name": "Rectangular 3x4", "A": [[1, 2, 1, 3], [2, 4, 0, 4], [3, 6, 1, 7]], "exp_rank": 2},
    {"name": "Tall Matrix 4x2", "A": [[1, 0], [0, 1], [1, 1], [2, 2]], "exp_rank": 2}
]

for case in rb_test_suite:
    rank, col_s, row_s, null_s = rank_and_basis(case["A"])
    assert rank == case["exp_rank"], f"❌ {case['name']}: Rank sai! (Got {rank}, Exp {case['exp_rank']})"
    
    # Kiểm tra Null Space: A * null_vector = 0
    for vec in null_s:
        assert np.allclose(np.dot(np.array(case["A"]), np.array(vec)), 0, atol=1e-7)
        
    print(f"   [OK] {case['name']}: Rank = {rank}, Null Space Verified.")

--- Testing Rank & Basis Suite (5+ Cases) ---
   [OK] Full Rank 3x3: Rank = 3, Null Space Verified.
   [OK] Rank 1 (All dependent): Rank = 1, Null Space Verified.
   [OK] Zero Matrix 2x2: Rank = 0, Null Space Verified.
   [OK] Rectangular 3x4: Rank = 2, Null Space Verified.
   [OK] Tall Matrix 4x2: Rank = 2, Null Space Verified.
